# Section 8 - Operate with OpenQASM

## Certification weight: 6%

## Objectives 
- Structure types in OpenQASM 3 programs
- Interpret OpenQASM semantics
- Interoperate different versions of OpenQASM with Qiskit
- Interact with the Qiskit IBM Runtime REST API

## Why OpenQASM?

Open Quantum Assembly Language is an imperative programming language for describing quantum circuit. It can be also used to characterize, validate, or debug quantum processors

OpenQASM is a standardized representation for quantum circuits, it's not an assembly language (Qiskit doesn't use OpenQASM during the transpilaiton process)

Although programming with python and notebooks can be convinient to thinker and test things, working with OpenQASM can be more efficient, and this language can be used in other frameworks, so the program will be portable.

## OpenQASM2 Minimal Syntax Elements

Here is an exemple of a program:
```    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c; 
```

Let's break down the program:

### Version Declaration & header inclusion
```
OPENQASM 2.0;
```

Declares the language and the version used. This version declaration is mandatory.

```
include "qelib1.inc";
```

This header contains the definitions of the common gates, such as `x`, `y`, `z`, `h`, `s`, `t`, `cx`, `swap`, rotational gates, ...
The gates are not included in the language itself, so it has to be included explicitly.


### Quantum and Classical Registers

```
qreg q[2];
creg c[2];
```

`qreg` declares a quantum register, while `creg` c lassical one.
One declared, the different qubits and cbits can be accessed with q[0], q[1], c[0] and c[1]

Those instructions does not initialize any quantum state. It just declares the logical qubits



### Gate Operations
```
h q[0];
cx q[0], q[1];
```

The operations are executed sequentially

It applies an Hadamard hates to qbit 0, and a CX to the qubit1, controlled by qubit 0

Other examples are the swap gate: `swap q[0], q[1];`, rotational gates: `rz(pi/4) q[0]; `


### Measurement
```
measure q[0] -> c[0];
```

Performs a quantum measurement. Qubit 0 will be measured, and its result stored in c[0] 

The syntax is: `measure qubit -> classical_bit;`


Those elements are not part of the example, but they are interesting to know:
### Barriers
Barriers can be added in a circuit with `barrier q;`. All the qubit of the register will have the barrier. 
It's possible to add a barrier to a single qubit by mentioning it: ``barrier q[1];`



### Custom Gates

Like funtions in programming languages, it's possible to create your own. For example:

```
gate bell a,b {
    h a;
    cx a,b;
}
```

Then, call it using `bell q[2], q[3];`



## Operating with OpenQASM2 in Qiskit

https://quantum.cloud.ibm.com/docs/en/guides/interoperate-qiskit-qasm2

It's possible to run QASM instructions in a python program, and save/restore a circuit from OpenQASM:




### Using OpenQASM instructions

You can create a circuit from OpenQASM instructions using `QuantumCircuit.from_qasm_str(<str>)` (works only for QASM2) or use `qiskit.qasm2.loads(<str>)`:

In [14]:
from qiskit import QuantumCircuit

qasm_program = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];
creg c[2];

h q[0];
cx q[0], q[1];

barrier q;

measure q[0] -> c[0];
measure q[1] -> c[1];
"""

qc = QuantumCircuit.from_qasm_str(qasm_program)
qc.draw()

┌───┐      ░ ┌─┐   
q_0: ┤ H ├──■───░─┤M├───
     └───┘┌─┴─┐ ░ └╥┘┌─┐
q_1: ─────┤ X ├─░──╫─┤M├
          └───┘ ░  ║ └╥┘
c: 2/══════════════╩══╩═
                   0  1

In [11]:
import qiskit.qasm2 

qc2 = qiskit.qasm2.loads(qasm_program)
qc2.draw()

┌───┐      ░ ┌─┐   
q_0: ┤ H ├──■───░─┤M├───
     └───┘┌─┴─┐ ░ └╥┘┌─┐
q_1: ─────┤ X ├─░──╫─┤M├
          └───┘ ░  ║ └╥┘
c: 2/══════════════╩══╩═
                   0  1

Instead of including a text blob in the python program, a program contained in a file can be included using `qc = qiskit.qasm2.load("program.qasm")`

## Exporting QASM2 from Qiskit

A `QuantumCircuit` object can be converted into QASM2 instructions using `qiskit.qasm2.dumps(<qc>)` to dump to a string, or `qiskit.qasm2.dump(<qc>)` to create a file

This can be used to serialize a circuit, or export it into a textfile



In [5]:
import qiskit.qasm2

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print(qiskit.qasm2.dumps(qc))

OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];


In [8]:
# Save the circuit into a textfile

with open("qc.qasm2", "w") as f:
    qiskit.qasm2.dump(qc, f)

## OpenQASM3

Thsi version of QASM brings more features compared to QASM2.

It supports dynamic circuits (if/else, while, for loops) and it's closer to the hardware.

The same commands load, loads, dump, dumps are provided by qiskit.qasm3

In [12]:
import qiskit.qasm3

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print(qiskit.qasm3.dumps(qc))

OPENQASM 3.0;
include "stdgates.inc";
bit[2] c;
qubit[2] q;
h q[0];
cx q[0], q[1];
c[0] = measure q[0];
c[1] = measure q[1];



**OpenQASM 3 Syntax**

Some parts of the syntax is different regarding to OpenQASM2:

### Version Declaration & header inclusion
OPENQASM 3;
Declares the language and the version used. This version declaration is mandatory.

include "stdgates.inc";

### Registers declaration
qubit[2] q;
bit[2] c;

### Measurements
It looks like a affectation : we store in a classical register the measurement of a qubit
c[0] = measure q[0];

### Conditionals & dynamic circuits:

- Control Flow (Most Important New Feature)
  ```
   Conditional Execution
   if (c[0] == 1) {
   x q[1];
  ```
  In this case, we apply a X gate on q1 if the classical bit c0 = 1

- For loops
  ```
  for int i in [0:1] {
    h q[i];
  }
  ```
  This loop will add an hadamard gate on q0 and q1


- While loops:
  ```
  while (c[0] == 0) {

    // Flip the qubit
    x q[0];

    // Measure again
    c[0] = measure q[0];
   }
   ```
   This will flip q0 until 0 is measured
  
}

# Interacting with the IBM Runtime API

The IBM Quantum Platform exposes REST APIs that allow you to:
- Authenticate with an API key
- List available quantum backends
- Submit runtime jobs
- Monitor job execution
- Retrieve results
- Manage sessions

This is useful when:
- You do not want to use the Python SDK
- You need language-agnostic integrations
- You want to automate workflows from CI/CD, web apps, or microservices


## Authentication

The authentication uses the API Key, and provides a token. This token (BEARER_TOKEN) will be valid for an hour, and has to be provided in every other API call.


# Q & A 

## OpenQASM & Qiskit — Multiple Choice Questions

---

## Question 1

What is the primary role of OpenQASM?

- A. Simulating quantum hardware
- B. Describing quantum circuits in a standardized format
- C. Replacing Python in Qiskit
- D. Optimizing transpiled circuits

<details>
<summary>Answer</summary>

**Correct answer: B**

OpenQASM is mainly a standardized language used to describe and exchange quantum circuits.

</details>

---

## Question 2

Which object is mainly used by Qiskit to represent a quantum circuit internally?

- A. Backend
- B. OpenQASM
- C. QuantumCircuit
- D. PassManager

<details>
<summary>Answer</summary>

**Correct answer: C**

`QuantumCircuit` is Qiskit’s primary circuit representation object.

</details>

---

## Question 3

What happens immediately after creating a `QuantumCircuit`?

- A. The circuit is executed on hardware
- B. The circuit is automatically transpiled
- C. Qiskit stores a logical circuit representation
- D. OpenQASM 3 code is generated

<details>
<summary>Answer</summary>

**Correct answer: C**

Creating a `QuantumCircuit` only creates a logical representation of the circuit.

</details>

---

## Question 4

What is transpilation in Qiskit?

- A. Converting Python into OpenQASM
- B. Adapting circuits to backend constraints
- C. Measuring qubits
- D. Simulating quantum noise

<details>
<summary>Answer</summary>

**Correct answer: B**

Transpilation adapts logical circuits to specific hardware requirements.

</details>

---

## Question 5

Which of the following may happen during transpilation?

- A. SWAP insertion
- B. Gate decomposition
- C. Qubit routing
- D. All of the above

<details>
<summary>Answer</summary>

**Correct answer: D**

All these operations are common transpilation tasks.

</details>

---

## Question 6

What does the following OpenQASM 2 instruction declare?

```qasm
qreg q[3];
```

- A. A classical register with 3 bits
- B. A quantum register with 3 qubits
- C. Three independent circuits
- D. A backend configuration

<details>
<summary>Answer</summary>

**Correct answer: B**

`qreg` defines a quantum register.

</details>

---

## Question 7

What is the purpose of this line?

```qasm
include "qelib1.inc";
```

- A. It imports backend calibration data
- B. It loads standard gate definitions
- C. It enables transpilation
- D. It activates dynamic circuits

<details>
<summary>Answer</summary>

**Correct answer: B**

The file contains standard gate definitions commonly used in OpenQASM 2.

</details>

---

## Question 8

Which statement about OpenQASM is the MOST accurate?

- A. It is always the exact language executed by hardware
- B. It is mainly a standardized circuit representation language
- C. Qiskit internally stores everything as OpenQASM
- D. It replaces transpilation

<details>
<summary>Answer</summary>

**Correct answer: B**

OpenQASM is mainly a standardized representation and interchange format.

</details>

---

## Question 9

Which instruction measures a qubit into a classical bit in OpenQASM 2?

- A. `read q[0] -> c[0];`
- B. `measure q[0] -> c[0];`
- C. `observe q[0] -> c[0];`
- D. `capture q[0] -> c[0];`

<details>
<summary>Answer</summary>

**Correct answer: B**

This is the standard OpenQASM 2 measurement syntax.

</details>

---

## Question 10

What is one major improvement introduced in OpenQASM 3?

- A. Native GPU execution
- B. Dynamic circuit support
- C. Automatic error correction
- D. Elimination of measurements

<details>
<summary>Answer</summary>

**Correct answer: B**

OpenQASM 3 introduces richer classical control and dynamic circuit capabilities.

</details>

---

## Question 11

Which OpenQASM 3 feature enables operations based on measurement results?

- A. Pulse scheduling
- B. Qubit routing
- C. Conditional statements
- D. Register declarations

<details>
<summary>Answer</summary>

**Correct answer: C**

Conditional execution enables dynamic circuit behavior.

</details>

---

## Question 12

What is the MOST accurate relationship between `QuantumCircuit` and OpenQASM?

- A. OpenQASM always replaces `QuantumCircuit`
- B. OpenQASM and `QuantumCircuit` usually represent circuits at similar abstraction levels
- C. `QuantumCircuit` is generated from transpiled hardware pulses
- D. OpenQASM is always lower-level than transpilation

<details>
<summary>Answer</summary>

**Correct answer: B**

Both usually describe logical circuits before backend adaptation.

</details>

---

## Question 13

Why may a transpiled circuit differ significantly from the original circuit?

- A. Hardware connectivity and native gate constraints
- B. Python syntax corrections
- C. Automatic algorithm replacement
- D. Classical compiler optimization only

<details>
<summary>Answer</summary>

**Correct answer: A**

Hardware limitations often require gate decomposition and qubit routing.

</details>

---

## Question 14

Which statement about OpenQASM 3 ecosystem support is TRUE?

- A. Every backend fully supports all OpenQASM 3 features
- B. OpenQASM 3 completely replaced OpenQASM 2 everywhere
- C. Support is still evolving across platforms
- D. OpenQASM 3 is deprecated

<details>
<summary>Answer</summary>

**Correct answer: C**

Support for OpenQASM 3 varies across frameworks and hardware providers.

</details>

---

## Question 15

Which pipeline is the MOST accurate?

- A.
```text
Python → Hardware
```

- B.
```text
Python → OpenQASM → Hardware
```

- C.
```text
Python → QuantumCircuit/OpenQASM → Transpilation → Backend execution
```

- D.
```text
Python → Measurement → Hardware
```

<details>
<summary>Answer</summary>

**Correct answer: C**

Quantum circuits are first represented logically, then transpiled for backend execution.

</details>

## OpenQASM & Python/Qiskit Operations — Multiple Choice Questions

---

## Question 1

Which Python method is commonly used to create a `QuantumCircuit` from an OpenQASM string?

- A. `QuantumCircuit.load_qasm()`
- B. `QuantumCircuit.from_qasm_str()`
- C. `QuantumCircuit.import_qasm()`
- D. `QuantumCircuit.read_qasm()`

<details>
<summary>Answer</summary>

**Correct answer: B**

`QuantumCircuit.from_qasm_str()` imports a circuit from an OpenQASM string.

</details>

---

## Question 2

Which instruction applies a Hadamard gate in OpenQASM 2?

- A. `hadamard q[0];`
- B. `h q[0];`
- C. `H q[0];`
- D. `gate h q[0];`

<details>
<summary>Answer</summary>

**Correct answer: B**

`h q[0];` applies a Hadamard gate to qubit 0.

</details>

---

## Question 3

Which OpenQASM 2 instruction applies a controlled-X gate?

- A. `cx q[0], q[1];`
- B. `cnot q[0], q[1];`
- C. `xctrl q[0], q[1];`
- D. `ctrlx q[0], q[1];`

<details>
<summary>Answer</summary>

**Correct answer: A**

`cx` is the standard controlled-X gate instruction.

</details>

---

## Question 4

What is the purpose of the following OpenQASM 2 instruction?

```qasm
barrier q;
```

- A. It resets all qubits
- B. It prevents certain compiler optimizations
- C. It measures all qubits
- D. It initializes the backend

<details>
<summary>Answer</summary>

**Correct answer: B**

A barrier acts as a transpiler/compiler boundary.

</details>

---

## Question 5

Which OpenQASM 3 declaration is valid?

- A. `qreg q[2];`
- B. `qubit[2] q;`
- C. `quantum q[2];`
- D. `register qubit q[2];`

<details>
<summary>Answer</summary>

**Correct answer: B**

OpenQASM 3 uses `qubit[2] q;`.

</details>

---

## Question 6

Which OpenQASM 3 syntax correctly stores a measurement result?

- A. `measure q[0] -> c[0];`
- B. `c[0] = q[0];`
- C. `c[0] = measure q[0];`
- D. `measure(q[0], c[0]);`

<details>
<summary>Answer</summary>

**Correct answer: C**

OpenQASM 3 uses assignment-style measurement syntax.

</details>

---

## Question 7

What is the main advantage of OpenQASM being text-based?

- A. Faster quantum execution
- B. Easier circuit sharing and inspection
- C. Automatic hardware optimization
- D. Removal of transpilation

<details>
<summary>Answer</summary>

**Correct answer: B**

Text-based representations are portable and human-readable.

</details>

---

## Question 8

Which Python instruction exports a circuit to OpenQASM 2 in many Qiskit versions?

- A. `qc.export_qasm()`
- B. `qc.to_qasm()`
- C. `qc.qasm()`
- D. `qc.dumps_qasm()`

<details>
<summary>Answer</summary>

**Correct answer: C**

`qc.qasm()` has historically been used to export OpenQASM 2.

</details>

---

## Question 9

Which statement about OpenQASM 2 is TRUE?

- A. It has extensive classical programming support
- B. It is mainly designed for static circuits
- C. It replaces transpilation
- D. It directly controls microwave pulses

<details>
<summary>Answer</summary>

**Correct answer: B**

OpenQASM 2 mainly describes static gate-based circuits.

</details>

---

## Question 10

Which OpenQASM 3 feature enables loops?

- A. `switch`
- B. `repeat`
- C. `for`
- D. `iterate`

<details>
<summary>Answer</summary>

**Correct answer: C**

OpenQASM 3 introduces loop constructs like `for`.

</details>

---

## Question 11

What is the role of classical bits in quantum circuits?

- A. They store measurement results
- B. They increase qubit coherence
- C. They replace qubits during transpilation
- D. They simulate quantum states

<details>
<summary>Answer</summary>

**Correct answer: A**

Classical bits are mainly used to store measurement outcomes.

</details>

---

## Question 12

Which Python object usually results from importing an OpenQASM file into Qiskit?

- A. Backend
- B. QuantumCircuit
- C. PassManager
- D. TranspilerLayout

<details>
<summary>Answer</summary>

**Correct answer: B**

Qiskit converts imported OpenQASM into a `QuantumCircuit`.

</details>

---

## Question 13

Why are custom gates useful in OpenQASM?

- A. They permanently modify hardware
- B. They allow reusable abstractions
- C. They bypass transpilation
- D. They automatically optimize circuits

<details>
<summary>Answer</summary>

**Correct answer: B**

Custom gates help organize and reuse circuit logic.

</details>

---

## Question 14

What is a key conceptual difference between OpenQASM 2 and OpenQASM 3?

- A. OpenQASM 3 supports richer classical control flow
- B. OpenQASM 2 supports dynamic circuits better
- C. OpenQASM 3 removes measurements
- D. OpenQASM 2 supports timing instructions better

<details>
<summary>Answer</summary>

**Correct answer: A**

OpenQASM 3 introduces more advanced classical programming features.

</details>

---

## Question 15

Why might a developer inspect OpenQASM generated from a circuit?

- A. To debug or understand low-level circuit structure
- B. To avoid using Qiskit
- C. To replace hardware calibration
- D. To bypass measurements

<details>
<summary>Answer</summary>

**Correct answer: A**

Inspecting OpenQASM helps understand how circuits are represented.

</details>